In [1]:
year = 2001
month = 12

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap

### URLs

In [3]:
# Ufiles = "https://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Ufiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Vfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridV"
Wfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridW"
Tfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridT"
Sfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridS"
#mesh url
Zgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_zgr.nc"
Hgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_hgr.nc"

### Environment 

In [4]:
os.environ["NETRC"] = "/home/b/b383184/.netrc"

### Mesh

In [5]:
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Zgr

<xarray.Dataset> Size: 555MB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 26MB ...
    hdept         (t, y, x) float64 106MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 106MB ...
    e3w_ps        (t, y, x) float64 106MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t

In [6]:
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')
ds_Hgr

<xarray.Dataset> Size: 1GB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/21)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    glamt         (t, y, x) float32 53MB ...
    glamu         (t, y, x) float32 53MB ...
    ...            ...
    e1f           (t, y, x) float64 106MB ...
    e2t           (t, y, x) float64 106MB ...
    e2u           (t, y, x) float64 106MB ...
    e2v           (t, y, x) float64 106MB ...
    e2f           (t, y, x) float64 106MB ...
    ff            (t, y, x) float64 106MB ...
Attributes:
    file_name:            mesh_hgr.nc
    TimeStamp:            03/02/2016 10:24:55 -0000
    Unlimited_Dimension:  t

### Functions

In [7]:
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

x0, x1, y0, y1

# x0=int(ds_Hgr.x.where(ds_Hgr.glamt >= lon0).min())
# x1=int(ds_Hgr.x.where(ds_Hgr.glamt <= lon1).max())

# y0=int(ds_Hgr.y.where(ds_Hgr.gphit >= lat0).min())
# y1=int(ds_Hgr.y.where(ds_Hgr.gphit <= lat1).max())

(2305, 3565, 1374, 1873)

In [8]:
# ds_Zgr.isel(x=slice(x0,x1),y=slice(y0,y1),t=0).mbathy.plot()

In [9]:
import calendar
import datetime
from datetime import date

In [10]:
last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)

print(end_date.strftime("%Y-%m-%d"))

2001-12-31


In [11]:
import pandas as pd

def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

# Example
days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))

In [12]:
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file):

    from tqdm import tqdm
    import xarray as xr
    
    parts = []
    for tt in tqdm(range(len(days)//2)):
        da = (
            xr.open_dataset(url, engine="pydap")[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all = da_all.where(da_all!= 9.96921e+36,np.nan)    
    da_all.to_dataset(name=varname).drop_encoding().to_netcdf(output_file)
    print(f"Saved {output_file}")

In [13]:
# import numpy as np
# a=np.arange(0,len(days),2)
# b=np.arange(0+1,len(days),2)
# print(a)
# print(b)
# starts = days[a]
# ends = days[b]
starts = days[0::2]
ends = days[1::2].tolist()  
# ends = days[b].tolist()  
ends[-1] = days[-1]
ends

for tt in  range(len(days)//2):
    print('start_date '+str(starts[tt]))
    print('end_date '+str(ends[tt]))

start_date 2001-12-01 12:00:00
end_date 2001-12-02 12:00:00
start_date 2001-12-03 12:00:00
end_date 2001-12-04 12:00:00
start_date 2001-12-05 12:00:00
end_date 2001-12-06 12:00:00
start_date 2001-12-07 12:00:00
end_date 2001-12-08 12:00:00
start_date 2001-12-09 12:00:00
end_date 2001-12-10 12:00:00
start_date 2001-12-11 12:00:00
end_date 2001-12-12 12:00:00
start_date 2001-12-13 12:00:00
end_date 2001-12-14 12:00:00
start_date 2001-12-15 12:00:00
end_date 2001-12-16 12:00:00
start_date 2001-12-17 12:00:00
end_date 2001-12-18 12:00:00
start_date 2001-12-19 12:00:00
end_date 2001-12-20 12:00:00
start_date 2001-12-21 12:00:00
end_date 2001-12-22 12:00:00
start_date 2001-12-23 12:00:00
end_date 2001-12-24 12:00:00
start_date 2001-12-25 12:00:00
end_date 2001-12-26 12:00:00
start_date 2001-12-27 12:00:00
end_date 2001-12-28 12:00:00
start_date 2001-12-29 12:00:00
end_date 2001-12-31 12:00:00


### Cut the meshes

In [14]:
!pwd

/work/bk1450/b383184/Amazon/Mercator/notebooks


In [15]:
# nZGR = ds_Zgr.isel(x=slice(x0, x1), y=slice(y0, y1))
# nZGR.to_netcdf('Zgr_cmesh')

In [16]:
# nHgr = ds_Hgr.isel(x=slice(x0, x1), y=slice(y0, y1))
# nHgr.to_netcdf('../data/Hgr_cmesh.nc')

### Data download

In [17]:
U_out = f'U_{start_date.strftime("%Y-%m-%d")[:7]}.nc'
# V_out = f'V_{start_date.strftime("%Y-%m-%d")[:7]}.nc'
# W_out = f'W_{start_date.strftime("%Y-%m-%d")[:7]}.nc'
# T_out = f'T_{start_date.strftime("%Y-%m-%d")[:7]}.nc'
# S_out = f'S_{start_date.strftime("%Y-%m-%d")[:7]}.nc'

# outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables/'
outpath = '/work/bk1450/b383184/Amazon/Mercator/notebooks'

In [18]:

download_MERCATOR(
    Ufiles, "vozocrtx", starts, ends, x0, x1, y0, y1,outpath+U_out
)

  0%|          | 0/15 [03:08<?, ?it/s]


ConnectionError: HTTPSConnectionPool(host='tds.mercator-ocean.fr', port=443): Read timed out.

In [ ]:
# download_MERCATOR(
#     Vfiles, "vomecrty", starts, ends, x0, x1, y0, y1,outpath+V_out 
# )

In [ ]:
# download_MERCATOR(
#     Wfiles, "vovecrtz", starts, ends, x0, x1, y0, y1,outpath+W_out 
# )

In [ ]:
# download_MERCATOR(
#     Tfiles, "votemper", starts, ends, x0, x1, y0, y1,outpath+T_out
# )

In [ ]:
# download_MERCATOR(
#     Sfiles, "vosaline", starts, ends, x0, x1, y0, y1,outpath+S_out 
# )

In [ ]:
# ds_U = (
#     xr.open_dataset(Ufiles, engine="pydap")
#     .sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1))
#     .sel(time_counter=slice('2004-02-01','2004-02-02'))
#     .vozocrtx
# )
# ds_U

In [ ]:
# ds_U = open_subset_var(Ufiles, "vozocrtx", x0, x1+1, y0, y1+1, start_date, end_date, engine="pydap")
# ds_V = open_subset_var(Vfiles, "vomecrty", x0, x1+1, y0, y1+1, start_date, end_date, engine="pydap")
# ds_W = open_subset_var(Wfiles, "vovecrtz", x0, x1+1, y0, y1+1, start_date, end_date, engine="pydap")
# ds_T = open_subset_var(Tfiles, "votemper", x0, x1+1, y0, y1+1, start_date, end_date, engine="pydap")
# ds_S = open_subset_var(Sfiles, "vosaline", x0, x1+1, y0, y1+1, start_date, end_date, engine="pydap")

In [ ]:
# from tqdm import tqdm

# parts = []
# for tt in  tqdm(range(len(days)//2)):
#     U = (
#         xr.open_dataset(Ufiles, engine="pydap",
#                         mask_and_scale=False, decode_cf=True)["vozocrtx"]
#         .sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1))
#         .sel(time_counter=slice(starts[tt],ends[tt]))
#         .astype("float32")
#         .load()
#     )
#     parts.append(U)

# U_all = xr.concat(parts, dim="time_counter")
# U_all.to_dataset(name="vozocrtx").to_netcdf(f'U_{start_date.strftime("%Y-%m-%d")[:7]}.nc')

In [ ]:
import xarray as xr
aa = xr.open_dataset('/work/bk1450/b383184/Amazon/Mercator/data/variables_c/U_1993-02c.nc')
aa.isel(time_counter=10,deptht=10).vozocrtx.plot()


In [ ]:
import xarray as xr
aa = xr.open_dataset('../data/variables/W_1997-01.nc')
print(aa) 

In [ ]:
aa = xr.open_dataset('../data/variables/S_1997-01.nc')
aa 

In [ ]:
ds_Zgr = xr.open_dataset('../data/Zgr_cmesh2.nc')
ds_Zgr

In [ ]:
# aa.where(aa.vozocrtx!= 9.96921e+36).vozocrtx.isel(time_counter=0,depth=0).plot(robust=True)

In [ ]:
# ds_U.isel(time_counter=0,deptht=0).plot()

In [ ]:
# ds_V = xr.open_dataset(Vfiles, engine="pydap").sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1)).sel(time_counter=slice(start_date,end_date)).vomecrty
# ds_V

In [ ]:
# ds_W = xr.open_dataset(Wfiles, engine="pydap").sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1)).sel(time_counter=slice(start_date,end_date)).vovecrtz
# ds_W

In [ ]:
# ds_T = xr.open_dataset(Tfiles, engine="pydap").sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1)).sel(time_counter=slice(start_date,end_date)).votemper
# ds_T

In [ ]:
# ds_S = xr.open_dataset(Sfiles, engine="pydap").sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1)).sel(time_counter=slice(start_date,end_date)).vosaline
# ds_S

In [ ]:
# ds_S.isel(time_counter=0,deptht=0).where(ds_S.isel(time_counter=0,deptht=0) !=9.96920997e+36,np.nan).plot() 

In [ ]:
# da = ds_S
# fill = getattr(da, "_FillValue", 9.969209968386869e+36)
# da = da.where(da != fill)  # keep valid values, others become NaN
# da.plot()

In [ ]:
# ds_S.nbytes / 1024**3  # GB actually held in memory